#### Imports

In [2]:
%run ../UtilsNew.ipynb

In [3]:
from scipy import stats
import statsmodels.tsa.stattools as ts
from pandas.plotting import autocorrelation_plot
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import statsmodels.api as sm
from time import time
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from math import sqrt

In [4]:
set_plot_size(15, 3)

In [144]:
def apply_sarima(df, order, seasonal_order):
    model = SARIMAX(df, order=order, seasonal_order=seasonal_order, enforce_stationarity=False)

    model_fit = model.fit(trend="nc", disp=False) #Vericiar o que é o nc

    return model_fit

def predict(model_fit, df_to_model, col, start, end):
    df_to_model['forecast'] = model_fit.predict(start=start,end=end, dynamic=True)

def get_residual_with_metrics(df):
   
    df_residuos = df[~df["forecast"].isna()]
    df_residuos["diff"] = df_residuos["radiacao"] - df_residuos["forecast"]

    mse = mean_squared_error(df_residuos["radiacao"], df_residuos["forecast"])
    mae = mean_absolute_error(df_residuos["radiacao"], df_residuos["forecast"])
    r2 = r2_score(df_residuos["radiacao"], df_residuos["forecast"])

    return {"mse": mse, "rmse": sqrt(mse), "mae":mae, "r2":r2}

def apply_config_model(order, seasonal_order, results, station, config):

    df_gp = load_and_filter(station, 7, 18, True)[["radiacao"]]
    
    model_fit = apply_sarima(df_gp, order, seasonal_order)

    start_pred = int(len(anual_df) * 0.9)
    end_pred = len(anual_df)
    
    predict(model_fit, df_gp, "radiacao", start=start_pred-250, end=end_pred-250)
    metrics = get_residual_with_metrics(df_gp)
    
    results[station][config] = metrics

def process_sarima(results):
    lista_estacoes = ["campo_grande", "curitiba", "manaus", "minas_gerais_gv", "mossoro", "pelotas", "brasilia", "minas_gerais_bh"] 
    # lista_estacoes = ["minas_gerais_gv"] 

    configs = {}
    configs[0] = [(1, 0, 0), (1, 0, 1, 12)]
    configs[1] = [(0, 0, 0), (1, 0, 1, 12)]
    configs[2] = [(1, 0, 1), (1, 0, 1, 12)]

    for estacao in lista_estacoes:
        print("Starting process for station " + estacao)
        results[estacao] = {}
        for config in configs.keys():
            apply_config_model(configs[config][0], configs[config][1], results, estacao, config)


In [145]:
results = {}
process_sarima(results)

Starting process for station campo_grande
Starting process for station curitiba
Starting process for station manaus
Starting process for station minas_gerais_gv
Starting process for station mossoro
Starting process for station pelotas
Starting process for station brasilia
Starting process for station minas_gerais_bh


In [146]:
def create_table_results_sarima(results):
    records = []
    for cidade, valores in results.items():
        for idx, metricas in valores.items():
            record = {'cidade': cidade, 'config': idx}
            record.update(metricas)
            records.append(record)
    
    df = pd.DataFrame(records)
    df.set_index('cidade', inplace=True)
    
    return df

In [163]:
pd.set_option('display.float_format', '{:.3f}'.format) 
create_table_results_sarima(results)

,config,mse,rmse,mae,r2
cidade,,,,,
campo_grande,0,554466.095,744.625,559.367,0.545
campo_grande,1,1036329.252,1018.003,767.708,0.150
campo_grande,2,552525.813,743.321,556.082,0.547
curitiba,0,2460518.222,1568.604,1193.568,-1.141
curitiba,1,2280418.672,1510.106,1142.338,-0.984
curitiba,2,2056591.548,1434.082,1081.087,-0.789
manaus,0,871253.450,933.410,668.657,0.008
manaus,1,496568.776,704.676,511.878,0.435
manaus,2,721473.851,849.396,613.683,0.179


In [161]:
results["minas_gerais_bh"]

{0: {'mse': 488624.8905272051,
  'rmse': 699.0170888663632,
  'mae': 586.9808416801529,
  'r2': 0.6377948282906042},
 1: {'mse': 517875.2728201915,
  'rmse': 719.635513868091,
  'mae': 607.0181400491191,
  'r2': 0.6161122657638254},
 2: {'mse': 488428.7128717967,
  'rmse': 698.8767508450948,
  'mae': 586.8165360598534,
  'r2': 0.6379402497841472}}